In [1]:
import pandas as pd
import os
from plotnine import *

import numpy as np
from pyensembl import EnsemblRelease
from matplotlib import gridspec, rc, rcParams
import matplotlib.pyplot as plt
import patchworklib as pw


#Set working directory
working_dir="/projects/b1042/AmaralLab/Maalavika/TF_Positional_Distribution_Model/data/Goode_ESC/"
os.chdir(working_dir)

<Figure size 72x72 with 0 Axes>

## For mm10 reference genome, compile list of peaks and edges for varying p-values

In [2]:
os.chdir('mm10/')

peaks = []
edges = []
for i in next(os.walk('.'))[1]:
    os.chdir(i)
    files = os.listdir()
    files = [f for f in files if ('peaks.narrowPeak' in f) & ('targets' not in f)] 
    for file in files:
        with open(file, 'r') as fp:
            x = len(fp.readlines())
            peaks.append([file.split('_')[0], file.split('_')[1], i, "All", x ])
        if x>0:
            df = pd.read_table(file + '_targets_10kbp.bed', header = None )
            edges.append([file.split('_')[0], file.split('_')[1], float(i),'10k', len(df[3].unique()) ])
            df = pd.read_table(file + '_targets_cds.bed', header = None )
            edges.append([file.split('_')[0], file.split('_')[1], float(i), 'cds', len(df[3].unique()) ])
        else:
            edges.append([file.split('_')[0], file.split('_')[1], float(i), '10k', x ])
            edges.append([file.split('_')[0], file.split('_')[1], float(i), 'cds', x ])
    os.chdir('../')
peaks = pd.DataFrame(peaks, columns = ['CellType', 'TF','pval', 'Flank', 'NumberOfPeaks'])
edges = pd.DataFrame(edges, columns = ['CellType', 'TF','pval',  'Flank', 'NumberOfEdges'])
peaks.to_csv('CompiledUnreplicatedPeaks.csv')
edges.to_csv('CompiledUnreplicatedEdges.csv')

## For mm39 reference genome, compile list of peaks(all) and edges (within 2kb of TSS), p =0.00001

In [3]:
os.chdir('../mm39/')
peaks_mm39 = []
files = os.listdir()
files = [f for f in files if 'peaks.narrowPeak' in f]
for file in files:
    with open(file, 'r') as fp:
        x = len(fp.readlines())
        peaks_mm39.append([file.split('_')[0], file.split('_')[1].upper(),1e-5 ,'All', x])
peaks_mm39 = pd.DataFrame(peaks_mm39, columns = ['CellType', 'TF', 'pval', 'Flank', 'NumberOfPeaks'])
peaks_mm39['Genome'] = 'mm39'
peaks['Genome']= 'mm10'
peaks_mm39 = pd.concat([peaks_mm39, peaks.loc[peaks.pval == '1e-5' ]])

edges_mm39 = []
files = os.listdir()
files = [f for f in files if 'targets_cds' in f]
for file in files:
    with open(file, 'r') as fp:
        x = len(fp.readlines())
        peaks_mm39.append([file.split('_')[0], file.split('_')[1].upper(),1e-5 ,'All', x])
        if file.split('_')[0] in ['HB', 'HE','HP','MAC','MES']:
            if x>0:
                df = pd.read_table(file , header = None )
                edges_mm39.append([file.split('_')[0], file.split('_')[1], 1e-5,'10k', len(df[3].unique()) ])
            else:
                edges_mm39.append([file.split('_')[0], file.split('_')[1],  1e-5, '10k', x ])
edges_mm39 = pd.DataFrame(edges_mm39, columns = ['CellType', 'TF', 'pval', 'Flank', 'NumberOfEdges'])
edges_mm39['Genome'] = 'mm39'
edges['Genome']= 'mm10'
edges_mm39 = pd.concat([edges_mm39, edges.loc[(edges.pval == 1e-5 ) & (edges.Flank =='10k') ]])

peaks_mm39.to_csv('CompiledUnreplicatedPeaks.csv')
edges_mm39.to_csv('CompiledUnreplicatedEdges.csv')

## For mm10 reference genome, p=0.00001 compile number of peaks and edges for various interaction distance

In [55]:
os.chdir(working_dir)
os.chdir('mm10')
#Reference tSS for all genes
ref = pd.read_table('TF_TSS_10kbp_named_mm10.bed')
ref['TSS'] = ref['end'] - 10000

#Get list of files (those identifying peaks within 10kb of gene)
os.chdir('1e-5')
files = os.listdir()
files = [f for f in files if 'targets_10kbp.bed' in f] 

peak_stats = pd.DataFrame()
edges_stats = pd.DataFrame()

for file in files:
    with open(file, 'r') as fp:
        x = len(fp.readlines())
    if x>0:
        df = pd.read_table(file,names = ['chr','start','end','gene'])
        df = df.merge(ref[['gene','TSS']], on='gene')
        df['min_dist'] = df.apply(lambda x: (abs(x['TSS'] - x['start'])+abs(x['TSS'] - x['end']))/2 ,axis =1)
        df['min_min_dist'] = df.groupby('gene')['min_dist'].transform(min)
        df_edges= df.loc[df.min_dist == df.min_min_dist]
               
        #Compile and estimate number of peaks in 1kb,2kb...10kb, cds flanks
        peak_num = [[file.split('_')[0],file.split('_')[1],1e-5, flank, 
                     sum(df['min_dist'] <=flank)] for flank in range(0,11000,1000)]
        peak_num = pd.DataFrame(peak_num, columns = ['CellType','TF', 'pval','Flank','NumberOfEdges'])
        peak_stats=pd.concat([peak_stats,peak_num])

        #Compile and estimate number of unique edges in 1kb,2kb...10kb, cds flanks
        edges_num = [[file.split('_')[0],file.split('_')[1],1e-5, flank, sum(df_edges['min_dist'] <=flank)] for flank in range(0,11000,1000)]
        edges_num = pd.DataFrame(edges_num, columns = ['CellType','TF', 'pval','Flank','NumberOfEdges'])
        edges_stats=pd.concat([edges_stats,edges_num])


peak_stats.to_csv('CompiledUnreplicatedPeaks.csv')
edges_stats.to_csv('CompiledUnreplicatedEdges.csv')


## Comparing list of edges found by us for interaction distance = coding sequence  

In [25]:
os.chdir(working_dir) 

genelist = ['Runx1', 'Cebpb','Sfpi1','Pou5f1', 'Tal1', 'Meis1', 'Lmo2', 'Gata1', 'Gata2', 'Gfi1',
           'Gfi1b', 'Esrrb', 'Elk4', 'Fli1', 'Sox2', 'Nanog' ]


observed_interactions =[]
#Select cell types in decoded network
ref = pd.read_csv('HP_decodedNetwork.csv')

os.chdir('raw/')
files = os.listdir()
files = [f for f in files if 'cds' in f]
files = [f for f in files if f.split('_')[1] in ref.CellType.unique()]

#Extract interactions present in gene list
for file in files: 
    with open(file, 'r') as fp:
        x = len(fp.readlines())
    if x > 0:
        df = pd.read_table(file, names = ['chrom','start', 'end','gene'])
        interactions = df.loc[df.gene.isin(genelist)].gene.unique()
        observed_interactions+=[[file.split('_')[2], i,file.split('_')[1]] for i in interactions]
observed_interactions = pd.DataFrame(observed_interactions, columns = ['Source', 'Target', 'CellType']).apply( lambda x: x.str.upper())
#Relabel genes with alternate names
observed_interactions['Target'] = observed_interactions.Target.replace('SFPI1', 'SPI1')  

#Join to obtain list of edges intersecting
combined = ref.merge(observed_interactions, on=['Source','Target', 'CellType'], 
                   how='outer', indicator=True).drop_duplicates()

#Rename edge categores
combined['_merge'] = combined['_merge'].replace('left_only', 'Only Goode et al.').replace('right_only', 'Only us')
combined.to_csv('Table2.csv')

#Count edges in each category
counts= combined.groupby(['CellType', '_merge']).size().reset_index(name='count')
counts['_merge'] = pd.Categorical(counts['_merge'], categories = ['Only us', 'both', 'Only Goode et al.'])
counts.to_csv('processed_edge_counts.csv')

In [29]:

#Function to identify minimum distance between two intervals
def solve(i1, i2):
     # sort the two ranges such that the range with smaller first element
     # is assigned to x and the bigger one is assigned to y
     x, y = sorted((i1, i2))

     #now if x[1] lies between x[0] and y[0](x[1] != y[0] but can be equal to x[0])
     #then the ranges are not overlapping and return the differnce of y[0] and x[1]
     #otherwise return 0 
     if x[0] <= x[1] < y[0] and all( y[0] <= y[1] for y in (i1,i2)):
        return y[0] - x[1]
     else:
        return 0

#For each missing edge estimate minimum distance from the coding sequence of the gene
for idx,row in abs_edges.iterrows():
    f = [i for i in files if (row['Source'] in i.upper()) & (row['CellType'] in i.upper())]
    target_TSS = ref.loc[ref.gene.str.upper() == row['Target']]
    if len(f) >1 : 
        print('more than one raw files match. Selecting first file.')
    bed = pd.read_table(f[0], compression = 'gzip',
             names = ['chrom','start','end'])
    bed = bed.loc[bed['chrom'] == target_TSS['chrom'].values[0]]
    dist = bed.apply(lambda x: solve([x['start'], x['end']], [target_TSS['TSS'].values[0], target_TSS['TES'].values[0]]), axis = 1).min()
    min_dist.append(dist)
    
abs_edges['min_dist'] = min_dist
abs_edges.to_csv('Compiled_absent_edges.csv')